# 딥앙상블 부트스트랩 v2 로컬 학습 (Windows + NVIDIA RTX / CUDA)

`bootstrap_v2`(사람검증 1016장, §2.16+trust_review 860장 반영 후)로 medium config
앙상블 5개(seed 0~4)를 학습한다. 맥북(M4/MPS)에서 이미 같은 방식으로 1회차(156장
기준, `outputs/ensemble_bootstrap_v1/`) 돌려서 검증된 파이프라인 그대로 — 코드
자체는 CUDA/ROCm/MPS 어디서든 그대로 동작하게(§2.17) 짜여 있어서 여기 셀들도
거의 동일하고, ROCm 대신 표준 CUDA PyTorch를 설치하는 것만 다르다.

**사전 준비**
1. NVIDIA 드라이버 + CUDA 지원 PyTorch가 설치돼 있을 것(없으면 아래 셀이 설치함).
2. **`BASE_DIR`**(아래 cell)를 실제 이 프로젝트를 둘 로컬 경로로 바꿀 것.
3. `bootstrap_v2.zip`(427MB, Mac `fine-tune` 레포 루트에서 만들어둔 것)을
   `BASE_DIR` 바로 밑에 미리 복사해둘 것.
4. 끝나면 `outputs/ensemble_bootstrap_v1_seed{0..4}_best.pth`(+`best_ll.pth`) 10개
   파일을 다시 Mac의 `fine-tune/outputs/ensemble_bootstrap_v2/seed{N}/`로 가져올 것
   (USB/AirDrop/클라우드 등) — 폴더 구조는 §2.20에서 만든 `ensemble_bootstrap_v1`과
   동일하게 맞추면 기존 평가 스크립트(`scripts/eval/check_ensemble_vs_human_gt.py`
   등)의 `ENSEMBLE_DIR`만 바꿔서 바로 재사용 가능.

## 0. 환경 설정 (로컬 경로 + CUDA 확인)

In [ ]:
import os

# ↓↓↓ 실행 전에 반드시 실제 경로로 바꿀 것 ↓↓↓
BASE_DIR = r'C:\Users\YOUR_NAME\umk_twinlite_ensemble_v2'

WORK_DIR = os.path.join(BASE_DIR, 'work')
REPO_DIR = os.path.join(WORK_DIR, 'TwinLiteNetPlus')
DATA_DIR = os.path.join(WORK_DIR, 'bdd100k')  # BDD100K.py가 '../bdd100k' 상대경로로 참조 - 이름 그대로 유지

os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(BASE_DIR, exist_ok=True)
print('BASE_DIR:', BASE_DIR)
print('이 폴더 바로 밑에 bootstrap_v2.zip / pretrained/medium.pth를 미리 넣어둘 것')

In [ ]:
# CUDA 지원 PyTorch가 없으면 설치 (이미 있으면 이 셀은 건너뛰어도 됨)
# CUDA 버전은 nvidia-smi로 먼저 확인하고 필요하면 --index-url의 cu버전을 맞출 것
# (예: CUDA 12.1 -> cu121, CUDA 12.4 -> cu124)
!pip install --no-cache-dir torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

In [ ]:
import torch
print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
else:
    print('[경고] CUDA 인식 안 됨 - 드라이버/PyTorch 설치 다시 확인할 것')

## 1. TwinLiteNetPlus 원본 레포 clone

In [ ]:
%cd {WORK_DIR}
!git clone --depth 1 https://github.com/chequanghuy/TwinLiteNetPlus.git
%cd {REPO_DIR}

## 2. 나머지 의존성 설치

In [ ]:
!pip install -q albumentations opencv-python scikit-learn scikit-image scipy pillow tqdm timm matplotlib pyyaml onnx onnxruntime

## 3. Pretrained 가중치(medium.pth) 준비

In [ ]:
CONFIG = 'medium'

import os, shutil
os.makedirs(f'{REPO_DIR}/pretrained', exist_ok=True)

local_pretrained = os.path.join(BASE_DIR, 'pretrained', f'{CONFIG}.pth')
target_pretrained = f'{REPO_DIR}/pretrained/{CONFIG}.pth'

if os.path.isfile(local_pretrained):
    shutil.copy(local_pretrained, target_pretrained)
    print('로컬에서 복사:', target_pretrained)
else:
    print(f'{local_pretrained} 없음 - gdown 시도')
    !pip install -q gdown
    !gdown --folder "https://drive.google.com/drive/folders/1EqBzUw0b17aEumZmWYrGZmbx_XJqU-vz" -O {REPO_DIR}/pretrained

assert os.path.isfile(target_pretrained), f'{target_pretrained} 없음 - 수동으로 받아서 넣을 것'
print('pretrained 준비 완료:', target_pretrained)

## 4. 데이터 준비 (bootstrap_v2, 1016장 — 사람검증 corpus 전체)

In [ ]:
import os, glob, random, shutil, zipfile

LOCAL_ZIP = os.path.join(BASE_DIR, 'bootstrap_v2.zip')
LOCAL_BOOTSTRAP = os.path.join(BASE_DIR, 'bootstrap_v2')
if not os.path.isdir(LOCAL_BOOTSTRAP):
    assert os.path.isfile(LOCAL_ZIP), f'{LOCAL_ZIP} 없음 - Mac에서 만든 bootstrap_v2.zip을 {BASE_DIR}에 복사해둘 것'
    with zipfile.ZipFile(LOCAL_ZIP) as zf:
        zf.extractall(BASE_DIR)
    print('압축 해제 완료:', LOCAL_BOOTSTRAP)

SRC_IMG = os.path.join(LOCAL_BOOTSTRAP, 'images')
SRC_DA = os.path.join(LOCAL_BOOTSTRAP, 'da_masks')
SRC_LL = os.path.join(LOCAL_BOOTSTRAP, 'll_masks')
VAL_RATIO = 0.15
SEED = 42

names = sorted(os.path.splitext(os.path.basename(p))[0] for p in glob.glob(os.path.join(SRC_IMG, '*.png')))
assert names, f'{SRC_IMG}에 png가 없음'
for n in names:
    for src, ext in [(SRC_DA, '.png'), (SRC_LL, '.png')]:
        assert os.path.isfile(os.path.join(src, n + ext)), f'{n}{ext} 마스크가 {src}에 없음'

random.Random(SEED).shuffle(names)
n_val = max(1, int(len(names) * VAL_RATIO))
val_names, train_names = names[:n_val], names[n_val:]
print(f'전체 {len(names)}장 -> train {len(train_names)} / val {len(val_names)}')

if os.path.isdir(DATA_DIR):
    shutil.rmtree(DATA_DIR)
for split, split_names in [('train', train_names), ('val', val_names)]:
    for sub in ['images', 'drivable_area_annotations', 'lane_line_annotations']:
        os.makedirs(os.path.join(DATA_DIR, sub, split), exist_ok=True)
    for n in split_names:
        shutil.copy(os.path.join(SRC_IMG, n + '.png'), os.path.join(DATA_DIR, 'images', split, n + '.png'))
        shutil.copy(os.path.join(SRC_DA, n + '.png'), os.path.join(DATA_DIR, 'drivable_area_annotations', split, n + '.png'))
        shutil.copy(os.path.join(SRC_LL, n + '.png'), os.path.join(DATA_DIR, 'lane_line_annotations', split, n + '.png'))
print('정리 완료:', DATA_DIR)

## 5. 하이퍼파라미터 + 코드 패치 (Mac/Colab판과 완전히 동일한 로직, §2.17)

In [ ]:
import yaml

with open(f'{REPO_DIR}/hyperparameters/twinlitev2_hyper.yaml') as f:
    hyp = yaml.safe_load(f)

hyp['lr'] = hyp['lr'] * 0.1
hyp['prob_crop'] = 0.0
hyp['ll_loss_weight'] = 2.0
hyp['ll_lr_mult'] = 2.5

FINETUNE_HYP = f'{REPO_DIR}/hyperparameters/finetune_hyper.yaml'
with open(FINETUNE_HYP, 'w') as f:
    yaml.safe_dump(hyp, f)
print('저장:', FINETUNE_HYP)

# --- loss.py 패치: ll_loss_weight + target.type_as(output)(device-agnostic) ---
loss_path = f'{REPO_DIR}/loss.py'
with open(loss_path) as f:
    loss_src = f.read()

if 'self.ll_weight' in loss_src:
    print('loss.py 이미 패치돼 있음 -> 스킵')
else:
    old_init = '''        self.seg_tver_da = TverskyLoss(mode="multiclass", alpha=alpha1, beta=1-alpha1, gamma=gamma1, from_logits=True)
        self.seg_tver_ll = TverskyLoss(mode="multiclass", alpha=alpha2, beta=1-alpha2, gamma=gamma2, from_logits=True)
        self.seg_focal = FocalLossSeg(mode="multiclass", alpha=alpha3, gamma=gamma3)'''
    new_init = '''        self.seg_tver_da = TverskyLoss(mode="multiclass", alpha=alpha1, beta=1-alpha1, gamma=gamma1, from_logits=True)
        self.seg_tver_ll = TverskyLoss(mode="multiclass", alpha=alpha2, beta=1-alpha2, gamma=gamma2, from_logits=True)
        self.seg_focal = FocalLossSeg(mode="multiclass", alpha=alpha3, gamma=gamma3)
        self.ll_weight = hyp.get("ll_loss_weight", 1.0)'''

    old_sum = '''        tversky_loss,focal_loss=tversky_da_loss+tversky_ll_loss,focal_da_loss+ focal_ll_loss'''
    new_sum = '''        tversky_loss,focal_loss=tversky_da_loss+self.ll_weight*tversky_ll_loss,focal_da_loss+self.ll_weight*focal_ll_loss'''

    assert old_init in loss_src and old_sum in loss_src, 'loss.py TotalLoss 코드가 예상과 다름'
    loss_src = loss_src.replace(old_init, new_init).replace(old_sum, new_sum)

    old_target = '''    target = target.type(output.type())'''
    if old_target in loss_src:
        loss_src = loss_src.replace(old_target, '''    target = target.type_as(output)''')

    old_da = '''        _,seg_da= torch.max(seg_da, 1)
        seg_da=seg_da.cuda()

        _,seg_ll= torch.max(seg_ll, 1)
        seg_ll=seg_ll.cuda()'''
    new_da = '''        _,seg_da= torch.max(seg_da, 1)
        seg_da=seg_da.to(out_da.device)

        _,seg_ll= torch.max(seg_ll, 1)
        seg_ll=seg_ll.to(out_ll.device)'''
    if old_da in loss_src:
        loss_src = loss_src.replace(old_da, new_da)

    with open(loss_path, 'w') as f:
        f.write(loss_src)
    print('loss.py 패치 완료')

# --- utils.py 패치: lr_mult 반영 + [:,12:-12] 크롭 제거 + device 일반화(.cuda()->.to(args.device)) ---
utils_path = f'{REPO_DIR}/utils.py'
with open(utils_path) as f:
    utils_src = f.read()

if "lr_mult" in utils_src and "[:,12:-12]" not in utils_src and "args.device" in utils_src:
    print('utils.py 이미 패치돼 있음 -> 스킵')
else:
    old_sched = '''def poly_lr_scheduler(args, hyp, optimizer, epoch, power=1.5):
    lr = round(hyp['lr'] * (1 - epoch / args.max_epochs) ** power, 8)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr
    return lr'''
    new_sched = '''def poly_lr_scheduler(args, hyp, optimizer, epoch, power=1.5):
    lr = round(hyp['lr'] * (1 - epoch / args.max_epochs) ** power, 8)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr * param_group.get('lr_mult', 1.0)
    return lr'''
    if old_sched in utils_src:
        utils_src = utils_src.replace(old_sched, new_sched)

    utils_src = utils_src.replace("        da_predict = da_predict[:,12:-12]\n", "")
    utils_src = utils_src.replace("        ll_predict = ll_predict[:,12:-12]\n", "")
    utils_src = utils_src.replace("        predict = predict[:,12:-12]\n", "")

    utils_src = utils_src.replace(
        "        if args.onGPU == True:\n            input = input.cuda().float() / 255.0        \n",
        "        if args.onGPU == True:\n            input = input.to(args.device).float() / 255.0\n")
    utils_src = utils_src.replace(
        "        with torch.cuda.amp.autocast():",
        "        with torch.autocast(device_type=args.device.type, enabled=(args.device.type == 'cuda')):")
    utils_src = utils_src.replace(
        "        input = input.cuda().half() / 255.0 if half else input.cuda().float() / 255.0",
        "        input = input.to(args.device).half() / 255.0 if half else input.to(args.device).float() / 255.0")

    with open(utils_path, 'w') as f:
        f.write(utils_src)
    print('utils.py 패치 완료')

# --- BDD100K.py 패치: letterbox -> plain resize(640x384) ---
bdd_path = f'{REPO_DIR}/BDD100K.py'
with open(bdd_path) as f:
    bdd_src = f.read()

if 'letterbox(image' not in bdd_src:
    print('BDD100K.py 이미 패치돼 있음 -> 스킵')
else:
    bdd_src = bdd_src.replace('image = letterbox(image, (H_, W_))', 'image = cv2.resize(image, (W_, H_))')
    bdd_src = bdd_src.replace('cv2.resize(label, (W_, 360))', 'cv2.resize(label, (W_, H_))')
    bdd_src = bdd_src.replace('cv2.resize(label1, (W_, 360))', 'cv2.resize(label1, (W_, H_))')
    bdd_src = bdd_src.replace('cv2.resize(label2, (W_, 360))', 'cv2.resize(label2, (W_, H_))')
    with open(bdd_path, 'w') as f:
        f.write(bdd_src)
    print('BDD100K.py 패치 완료 -> letterbox 제거, plain resize(640x384)')

# --- loss.py 크롭 제거 ---
with open(loss_path) as f:
    loss_src2 = f.read()
if '[:,:,12:-12]' not in loss_src2:
    print('loss.py 크롭 이미 제거돼 있음 -> 스킵')
else:
    loss_src2 = loss_src2.replace('out=outputs[:,:,12:-12]', 'out=outputs')
    loss_src2 = loss_src2.replace('out_da,out_ll=out_da[:,:,12:-12],out_ll[:,:,12:-12]', 'out_da,out_ll=out_da,out_ll')
    with open(loss_path, 'w') as f:
        f.write(loss_src2)
    print('loss.py 크롭 제거 완료')

## 6. 파인튜닝 스크립트 작성 (맥북 라운드1과 완전히 동일, `--seed` 지원)

In [ ]:
finetune_script = r'''
import os
import random
import torch
import torch.optim.lr_scheduler
import torch.backends.cudnn as cudnn
import numpy as np
import yaml
import math
from copy import deepcopy
from argparse import ArgumentParser

from model.model import TwinLiteNetPlus
from loss import TotalLoss
from utils import train, val, netParams, save_checkpoint, poly_lr_scheduler
import BDD100K

class ModelEMA:
    def __init__(self, model, decay=0.9999, updates=0):
        self.ema = deepcopy(model).eval()
        self.updates = updates
        self.decay = lambda x: decay * (1 - math.exp(-x / 2000))
        for p in self.ema.parameters():
            p.requires_grad_(False)

    def update(self, model):
        with torch.no_grad():
            self.updates += 1
            d = self.decay(self.updates)
            msd = model.state_dict()
            for k, v in self.ema.state_dict().items():
                if v.dtype.is_floating_point:
                    v *= d
                    v += (1. - d) * msd[k].detach()

def resolve_device():
    if torch.cuda.is_available():
        return torch.device('cuda')
    if torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')

def train_net(args, hyp):
    use_ema = args.ema
    device = resolve_device()
    args.device = device
    args.onGPU = device.type != 'cpu'
    print(f'[finetune] device: {device}')

    random.seed(args.seed)
    np.random.seed(args.seed)
    torch.manual_seed(args.seed)

    model = TwinLiteNetPlus(args)

    if args.weight and os.path.isfile(args.weight):
        state = torch.load(args.weight, map_location='cpu')
        if isinstance(state, dict) and 'state_dict' in state:
            state = state['state_dict']
        missing, unexpected = model.load_state_dict(state, strict=False)
        print(f'[finetune] pretrained 로드: {args.weight}')
        print(f'[finetune]   missing={len(missing)} unexpected={len(unexpected)}')
    else:
        print(f'[finetune] --weight 없음/파일 없음({args.weight}) - 랜덤 초기화로 진행')

    os.makedirs(args.savedir, exist_ok=True)

    g = torch.Generator()
    g.manual_seed(args.seed)
    trainLoader = torch.utils.data.DataLoader(
        BDD100K.Dataset(hyp, valid=False),
        batch_size=args.batch_size, shuffle=True, num_workers=args.num_workers,
        pin_memory=(device.type == 'cuda'), generator=g)

    valLoader = torch.utils.data.DataLoader(
        BDD100K.Dataset(hyp, valid=True),
        batch_size=args.batch_size, shuffle=False, num_workers=args.num_workers,
        pin_memory=(device.type == 'cuda'))

    model = model.to(device)
    if device.type == 'cuda':
        cudnn.benchmark = True

    print(f'Total network parameters: {netParams(model)}')

    criteria = TotalLoss(hyp)
    start_epoch = 0
    lr = hyp['lr']
    ll_lr_mult = hyp.get('ll_lr_mult', 1.0)
    ll_param_ids = set()
    ll_params = []
    for name, p in model.named_parameters():
        if '_ll' in name:
            ll_params.append(p)
            ll_param_ids.add(id(p))
    other_params = [p for p in model.parameters() if id(p) not in ll_param_ids]
    print(f'[finetune] ll 디코더 파라미터 {len(ll_params)}개 텐서 (LR x{ll_lr_mult}) / 나머지 {len(other_params)}개 텐서')
    optimizer = torch.optim.AdamW([
        {'params': other_params, 'lr': lr, 'lr_mult': 1.0},
        {'params': ll_params, 'lr': lr * ll_lr_mult, 'lr_mult': ll_lr_mult},
    ], betas=(hyp['momentum'], 0.999), eps=hyp['eps'], weight_decay=hyp['weight_decay'])

    ema = ModelEMA(model) if use_ema else None

    best_da_miou = -1.0
    best_ll_iou = -1.0
    if args.resume and os.path.isfile(args.resume):
        if args.resume.endswith('.tar'):
            print(f"=> Loading checkpoint '{args.resume}'")
            checkpoint = torch.load(args.resume, map_location='cpu')
            start_epoch = checkpoint['epoch']
            model.load_state_dict(checkpoint['state_dict'])
            model = model.to(device)
            if use_ema:
                ema.ema.load_state_dict(checkpoint['ema_state_dict'])
                ema.ema = ema.ema.to(device)
                ema.updates = checkpoint['updates']
            optimizer.load_state_dict(checkpoint['optimizer'])
            best_da_miou = checkpoint.get('best_da_miou', -1.0)
            best_ll_iou = checkpoint.get('best_ll_iou', -1.0)
            print(f"=> Loaded checkpoint (epoch {checkpoint['epoch']}, best_da_miou={best_da_miou:.3f}, best_ll_iou={best_ll_iou:.3f})")
        else:
            print(f"=> No valid checkpoint found at '{args.resume}'")

    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

    for epoch in range(start_epoch, args.max_epochs):
        model_file_name = os.path.join(args.savedir, f'model_{epoch}.pth')
        poly_lr_scheduler(args, hyp, optimizer, epoch)
        lr = optimizer.param_groups[0]['lr']
        ll_lr = optimizer.param_groups[1]['lr'] if len(optimizer.param_groups) > 1 else lr
        print(f'Learning rate: {lr} (ll decoder: {ll_lr})')

        model.train()
        ema = train(args, trainLoader, model, criteria, optimizer, epoch, scaler, args.verbose, ema if use_ema else None)

        model.eval()
        da_segment_results, ll_segment_results = val(valLoader, ema.ema if use_ema else model, args=args)

        print(f'Driving Area Segment: mIOU({da_segment_results[2]:.3f})')
        print(f'Lane Line Segment: Acc({ll_segment_results[0]:.3f}) IOU({ll_segment_results[1]:.3f})')

        torch.save(ema.ema.state_dict(), model_file_name) if use_ema else torch.save(model.state_dict(), model_file_name)
        if da_segment_results[2] > best_da_miou:
            best_da_miou = da_segment_results[2]
            best_path = os.path.join(args.savedir, 'best.pth')
            torch.save(ema.ema.state_dict() if use_ema else model.state_dict(), best_path)
            print(f'[finetune] 새 best(da) 저장: {best_path} (da mIoU={best_da_miou:.3f})')
        if ll_segment_results[1] > best_ll_iou:
            best_ll_iou = ll_segment_results[1]
            best_ll_path = os.path.join(args.savedir, 'best_ll.pth')
            torch.save(ema.ema.state_dict() if use_ema else model.state_dict(), best_ll_path)
            print(f'[finetune] 새 best(ll) 저장: {best_ll_path} (ll IOU={best_ll_iou:.3f})')

        save_checkpoint({
            'epoch': epoch + 1,
            'state_dict': model.state_dict(),
            'ema_state_dict': ema.ema.state_dict() if use_ema else None,
            'updates': ema.updates if use_ema else None,
            'optimizer': optimizer.state_dict(),
            'lr': lr,
            'best_da_miou': best_da_miou,
            'best_ll_iou': best_ll_iou,
        }, os.path.join(args.savedir, 'checkpoint.pth.tar'))

        if args.drive_backup_dir:
            import shutil as _shutil
            os.makedirs(args.drive_backup_dir, exist_ok=True)
            for _fn in ('checkpoint.pth.tar', 'best.pth', 'best_ll.pth'):
                _src = os.path.join(args.savedir, _fn)
                if os.path.isfile(_src):
                    _shutil.copy(_src, os.path.join(args.drive_backup_dir, _fn))
            print(f'[finetune] epoch {epoch} 체크포인트 백업 완료: {args.drive_backup_dir}')

if __name__ == '__main__':
    parser = ArgumentParser()
    parser.add_argument('--max_epochs', type=int, default=100)
    parser.add_argument('--num_workers', type=int, default=4)
    parser.add_argument('--batch_size', type=int, default=8)
    parser.add_argument('--savedir', default='./finetune_out')
    parser.add_argument('--hyp', type=str, default='./hyperparameters/finetune_hyper.yaml')
    parser.add_argument('--resume', type=str, default='')
    parser.add_argument('--weight', type=str, default='')
    parser.add_argument('--config', default='small')
    parser.add_argument('--verbose', action='store_true')
    parser.add_argument('--ema', action='store_true')
    parser.add_argument('--drive_backup_dir', type=str, default='')
    parser.add_argument('--seed', type=int, default=0, help='앙상블 멤버 구분용 시드 - 가중치 초기화/데이터 셔플 둘 다에 적용')
    args = parser.parse_args()

    with open(args.hyp, errors='ignore') as f:
        hyp = yaml.safe_load(f)

    train_net(args, hyp.copy())
'''

with open(f'{REPO_DIR}/finetune.py', 'w') as f:
    f.write(finetune_script)
print('작성 완료:', f'{REPO_DIR}/finetune.py')

## 7. 앙상블 5개 학습 실행 (seed 0~4 순차)

RTX GPU면 맥북(M4/MPS, 1epoch 약 74초)보다 훨씬 빠를 것 — 실제 소요시간 보고
`BATCH_SIZE`를 올려서 더 단축할 수 있는지 판단할 것(VRAM 여유 있으면).

In [ ]:
%cd {REPO_DIR}
MAX_EPOCHS = 40
BATCH_SIZE = 8

for SEED in range(5):
    SAVEDIR = os.path.join(BASE_DIR, f'finetune_out_ensemble_v2_seed{SEED}')
    print(f'=== seed {SEED} 시작 ===')
    train_cmd = (
        f'python finetune.py --config {CONFIG} --weight pretrained/{CONFIG}.pth '
        f'--hyp hyperparameters/finetune_hyper.yaml --max_epochs {MAX_EPOCHS} '
        f'--batch_size {BATCH_SIZE} --savedir "{SAVEDIR}" --seed {SEED} --verbose'
    )
    print(train_cmd)
    get_ipython().system(train_cmd)
    print(f'=== seed {SEED} 완료 ===')

print('앙상블 5개 전부 완료')

## 8. 완료 후 — Mac으로 파일 가져오기

`{BASE_DIR}/finetune_out_ensemble_v2_seed{0..4}/best.pth`(+`best_ll.pth`) 총 10개
파일을 Mac의 `fine-tune/outputs/ensemble_bootstrap_v2/seed{N}/`로 복사해올 것
(USB/AirDrop/클라우드 등). 그러면 §2.20에서 만든 평가 스크립트들의 `ENSEMBLE_DIR`을
`outputs/ensemble_bootstrap_v1` → `outputs/ensemble_bootstrap_v2`로만 바꿔서 바로
비교/검증할 수 있음.